## **Data Validation Notebook**

Purpose: This notebook runs validation code for the scraped data stored in `data/scraped/products.jsonl`. It validates which products are food, whether ingredients data are present for later NOVA classification, and the barcode matching process to be used with OpenFoodFacts.

### 1. Imports

In [1]:
import time
import json
import requests
import re

import pandas as pd
pd.set_option('display.max_columns', None)

from collections import Counter
from pathlib import Path

from tqdm.notebook import tqdm_notebook

PRODUCT_URL = "https://world.openfoodfacts.org/api/v2/product"

### 2. Inspect Scraped Data

#### 2.1 Count products by category

In [2]:
# Count top-level categories (index 1 in food_type, after "Groceries")
category_counts = Counter()
path = Path("/files/problem-set-1-grayusan/data/scraped/products.jsonl")

with path.open("r", encoding="utf-8") as f:
    for line in f:
        product = json.loads(line)
        categories = product.get("food_type") or []
        if len(categories) > 1:
            category_counts[categories[1]["name"]] += 1

print(f"Total products: {sum(category_counts.values())}")
print(f"Number of categories: {len(category_counts)}\n")

for category, count in category_counts.most_common():
    print(f"{category}: {count}")

Total products: 6938
Number of categories: 14

Fresh & Chilled: 2951
Food Cupboard: 2212
Frozen: 492
Bakery: 430
Toiletries, Health & Beauty: 217
Tea, Coffee & Soft Drinks: 209
Beer, Wine & Spirits: 145
All food to order: 90
Baby & Toddler: 85
Pet: 46
Household: 41
Dietary & Lifestyle: 17
Home: 2
Waitrose Brands: 1


#### 2.2 Create `df_waitrose` and inspect sample data

In [3]:
df_waitrose = pd.read_json('/files/problem-set-1-grayusan/data/scraped/products.jsonl', lines=True)
df_waitrose.head()

,name,barcode,url,food_type,pricing,ingredients
0,GOOD TO GO Pomegranate Seeds,"[5000169631362, 5000169158333]",https://www.waitrose.com/ecom/products/good-to...,"[{'id': '10051', 'name': 'Groceries'}, {'id': ...",1.65,INGREDIENTS: pomegranate seeds (100%).
1,Duchy Organic Smoked Salmon,[5000169173701],https://www.waitrose.com/ecom/products/duchy-o...,"[{'id': '10051', 'name': 'Groceries'}, {'id': ...",5.62,INGREDIENTS: Organic salmon (Salmo salar) (fis...
2,Natures Candy USA Medjool Dates,[5061048250005],https://www.waitrose.com/ecom/products/natures...,"[{'id': '10051', 'name': 'Groceries'}, {'id': ...",5.25,None
3,Essential Mild Cheddar S2,[5000169577851],https://www.waitrose.com/ecom/products/essenti...,"[{'id': '10051', 'name': 'Groceries'}, {'id': ...",2.75,INGREDIENTS: Cheddar Cheese (milk).
4,No.1 Keralan Inspired Fish Curry for 2,[5063210008216],https://www.waitrose.com/ecom/products/no1-ker...,"[{'id': '10051', 'name': 'Groceries'}, {'id': ...",6.50,"INGREDIENTS: Hake (fish) (52%), onion, coconut..."


### 3. Inspect OpenFoodFacts API Data

#### 3.1. Request OpenFoodFacts API for sample data

In [ ]:
openfoodfacts_responses = []

sample = df_waitrose['barcode'].head(5)

for idx, barcodes in tqdm_notebook(sample.items()):
    row_results = []
    for barcode in barcodes:
        response = requests.get(f'{PRODUCT_URL}/{barcode}', timeout=30)
        if response.status_code == 200:
            result = response.json()
            row_results.append({
                'product_id': idx,
                'barcode_used': barcode,
                'response': result,
            })
        time.sleep(0.5)

    if not row_results:
        openfoodfacts_responses.append({
            'product_id': idx,
            'barcode_used': barcodes[0] if barcodes else None,
            'response': None,
        })

    else:
        openfoodfacts_responses.extend(row_results)

with open("/files/problem-set-1-grayusan/data/enriched/off_data_validation_sample.json", "w") as f:
    json.dump(openfoodfacts_responses, f, indent=2, ensure_ascii=False)
print(f"Saved {len(openfoodfacts_responses)} responses to off_data_validation_sample.json")

#### 3.2. Load API response into a DataFrame and inspect

In [4]:
with open('off_data_validation_sample.json', 'r') as f:
    api_data = json.load(f)

In [5]:
df_api = pd.json_normalize(api_data, meta=['response', 'product'])
df_api

,product_id,barcode_used,response,response.code,response.product._id,response.product._keywords,response.product.added_countries_tags,response.product.allergens,response.product.allergens_from_ingredients,response.product.allergens_from_user,response.product.allergens_hierarchy,response.product.allergens_tags,response.product.amino_acids_prev_tags,response.product.amino_acids_tags,response.product.brands,response.product.brands_old,response.product.brands_tags,response.product.categories,response.product.categories_hierarchy,response.product.categories_lc,response.product.categories_old,response.product.categories_properties.agribalyse_food_code:en,response.product.categories_properties.agribalyse_proxy_food_code:en,response.product.categories_properties.ciqual_food_code:en,response.product.categories_properties_tags,response.product.categories_tags,response.product.checkers_tags,response.product.ciqual_food_name_tags,response.product.code,response.product.codes_tags,response.product.compared_to_category,response.product.complete,response.product.completeness,response.product.correctors_tags,response.product.countries,response.product.countries_debug_tags,response.product.countries_hierarchy,response.product.countries_lc,response.product.countries_tags,response.product.created_t,response.product.creator,response.product.data_quality_bugs_tags,response.product.data_quality_errors_tags,response.product.data_quality_info_tags,response.product.data_quality_tags,response.product.data_quality_warnings_tags,response.product.data_sources,response.product.data_sources_tags,response.product.ecoscore_data.adjustments.origins_of_ingredients.aggregated_origins,response.product.ecoscore_data.adjustments.origins_of_ingredients.epi_score,response.product.ecoscore_data.adjustments.origins_of_ingredients.epi_value,response.product.ecoscore_data.adjustments.origins_of_ingredients.origins_from_categories,response.product.ecoscore_data.adjustments.origins_of_ingredients.origins_from_origins_field,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_score,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.ad,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.al,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.at,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.ax,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.ba,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.be,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.bg,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.ch,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.cy,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.cz,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.de,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.dk,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.dz,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.ee,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.eg,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.es,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.fi,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.fo,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.fr,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_scores.gg,response.product.ecoscore_data.adjustments.origins_of_ingredients.transportation_s

In [6]:
df_api['response.product.nova_group']

0    NaN
1    NaN
2    NaN
3    NaN
4    3.0
Name: response.product.nova_group, dtype: float64

### 4. Inspect unique categories to filter non-food products

In [ ]:
categories = {}

with open("/files/problem-set-1-grayusan/data/scraped/products.jsonl") as f:
    for line in f:
        product = json.loads(line)
        for cat in (product.get("food_type") or [])[:3]:
            if isinstance(cat, dict):
                categories[cat["id"]] = cat["name"]
            else:
                print(f"Unexpected food_type entry: {cat}")

for id, name in sorted(categories.items(), key=lambda x: x[1]):
    print(f"{id}: {name}")

print(f"\n{len(categories)} unique categories found")

In [7]:
MUP_df = pd.read_excel('/files/problem-set-1-grayusan/data/enriched/MUP.csv')

In [8]:
BASE_DIR = Path.cwd().parent.parent
DATA_DIR = BASE_DIR / 'data'
SCRAPED_DIR = DATA_DIR / 'scraped'
ENRICHED_DIR = DATA_DIR / 'enriched'

scraped_file = SCRAPED_DIR / 'products.jsonl'
enriched_file = ENRICHED_DIR / 'products.jsonl'
scraped = pd.read_json(scraped_file, lines=True).to_dict(orient='records')

mup_df = pd.read_excel(ENRICHED_DIR / 'MUP.csv')

In [28]:
df_ingredients = df_waitrose[['name', 'ingredients']]
df_no_ingredients = df_ingredients[df_ingredients['ingredients'].isna()]
df_no_ingredients

,name,ingredients
2,Natures Candy USA Medjool Dates,None
6,Waitrose Blush Pears,None
15,Rodda's Classic Cornish Clotted Cream,None
18,Waitrose British Regal Red Potatoes,None
25,Graham's Dairy Gold Top Original Jersey Whole ...,None
...,...,...
6933,Waitrose Italian Style Instant Coffee,None
6940,No.1 Kenya Ground Coffee,None
6942,No.1 Java Ground Coffee,None
6947,Waitrose Gold Colombian Instant Coffee,None


In [36]:
from excluded_categories import EXCLUDED_CATEGORY_IDS

excluded = set(EXCLUDED_CATEGORY_IDS)

mask = df_waitrose['food_type'].apply(
    lambda cats: any(cat['id'] in excluded for cat in (cats or []))
)

dropped_products = mask.sum()
df_waitrose = df_waitrose.loc[~mask]

print(f'{dropped_products} non-food products dropped')

317 non-food products dropped


In [12]:
for product in scraped:
    ingredients = product.get('ingredients', [])

    print(f'{product.get('name', [])} Ingredients: {ingredients}')
    

GOOD TO GO Pomegranate Seeds Ingredients: INGREDIENTS: pomegranate seeds (100%).
Duchy Organic Smoked Salmon Ingredients: INGREDIENTS: Organic salmon (Salmo salar) (fish) (97%), sea salt, organic cane sugar
Natures Candy USA Medjool Dates Ingredients: None
Essential Mild Cheddar S2 Ingredients: INGREDIENTS: Cheddar Cheese (milk).
No.1 Keralan Inspired Fish Curry for 2 Ingredients: INGREDIENTS: Hake (fish) (52%), onion, coconut extract, rapeseed oil, mango purée, ginger purée, garlic purée, sliced red chilli, tomato paste, chopped coriander, paprika, green chilli purée, salt, coriander sprig, lemon juice, tamarind paste, mustard seeds, turmeric, cornflour, ground coriander, whole curry leaves, chilli powder, fennel                                                                                                                                                                                                                                                                                     

In [49]:
df_mup = pd.read_excel(ENRICHED_DIR / 'MUP.csv')
df_mup

,category,mup_id,mup_name,individual_compound,match_type,is_extended
0,cosmetic_additives__flavours,1,flavour,flavor,exact,False
1,cosmetic_additives__flavours,1,flavour,flavour,exact,False
2,cosmetic_additives__flavour_enhancers,2,e62*,e620 - e629,prefix,False
3,cosmetic_additives__flavour_enhancers,3,e63*,e630 - e637,prefix,False
4,cosmetic_additives__flavour_enhancers,4,e640,e640,exact,False
...,...,...,...,...,...,...
104,non_culinary__protein_sources,64,whey,whey solids,exact,True
105,non_culinary__fibres,65,fibre,fibre,exact,False
106,non_culinary__fibres,65,fibre,fibre isolate,exact,True
107,non_culinary__fibres,65,fibre,insoluble fibre,exact,False


In [71]:
df_mup = pd.read_excel(ENRICHED_DIR / 'MUP.csv.xlsx')
df_mup = df_mup.to_dict(orient='records')

def has_mup(ingredients):
    
    # Assign label to null ingredients

    if product['ingredients'] is None or product['ingredients'] == '': 
        nova_group = 'Not confirmed UPF'
        product['nova_group'] = nova_group
        return product

    # Define normalizer and normalize ingredients and df_mup

    def normalize(ingredients: str) -> str:
        return re.sub(r'[^a-zA-Z\s]', '', ingredients.lower())

    ingredients = normalize(product['ingredients'])
    
    exact_terms = set(
        df_mup.loc[df_mup['match_type'] == 'exact', 'mup_name'].str.lower().str.strip()
    )

    prefix_terms = tuple(
        df_mup.loc[df_mup['match_type'] == 'prefix', 'mup_name'].str.lower().str.strip()
    )

    # Identify if ingredients include / start with MUP

    return (
        any(ing in exact_terms for ing in ingredients)
        or
        any(ing.startswith(prefix_terms) for ing in ingredients)
    )

In [72]:
mask = df_waitrose['ingredients'].apply(has_mup)
df_waitrose.loc[mask, 'nova_group'] = 4
df_waitrose.loc[mask, 'nova_source'] = 'MUP present'

AttributeError: 'list' object has no attribute 'loc'

In [62]:
products = df_waitrose.to_dict(orient='records')
enriched_products = [generate_NOVA_classification(p) for p in products]

In [ ]:
if mup['mup_name'] in product_ingredients and mup['match_type'] == 'exact':
        nova_group = 4
        nova_source = 'MUP present'
        product['nova_group'] = nova_group
        product['nova_source'] = nova_source

    elif mup['mup_match']

    return product

KeyError: 'nova_group'

In [2]:
import requests
resp = requests.get(
    'https://world.openfoodfacts.org/api/v2/product/3329770045569',
    headers={'User-Agent': 'DS205-LSE/1.0'}
)
print(resp.status_code)

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))